# mini-LLM en GPU (Google Colab)

Entrena el perfil `configs/colab.yaml`: **29,5 M de parámetros**, 256 tokens de contexto, 1 000 000 de cuentos.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** → Guardar.

Ejecuta las celdas **en orden**. Todo lo importante (modelos, tokenizador, datos procesados) se guarda en tu
Google Drive, en la carpeta `mini-llm-colab`. Si Colab se desconecta, vuelve a ejecutar las celdas 1 a 4 y la de
entrenamiento: continúa desde el último punto guardado (cada 500 iteraciones).

Mantén esta pestaña abierta mientras entrena: Colab gratis corta las sesiones inactivas.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución -> Cambiar tipo -> GPU T4"
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name())

## 2. Conectar Google Drive (aquí se guarda todo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/mini-llm-colab'
!mkdir -p {DRIVE}/artifacts

## 3. Descargar el proyecto e instalarlo

In [ ]:
%cd /content
!test -d mini-llm || git clone -q https://github.com/carlosjaka54/mini-llm.git
%cd /content/mini-llm
!git pull -q
!pip install -q -e .
# Los artefactos (modelos, tokenizadores, métricas) viven en Drive: sobreviven a una desconexión
!test -L artifacts || (cp -rn artifacts/. {DRIVE}/artifacts/ && rm -rf artifacts && ln -s {DRIVE}/artifacts artifacts)
!ls -la artifacts

## 4. Datos (≈ 20-30 min la primera vez; luego se recuperan de Drive en 1 min)

Descarga 1 000 000 de cuentos, entrena el tokenizador BPE de 8 192 tokens y convierte todo en tokens.
Al terminar guarda una copia en Drive para no repetirlo si Colab se desconecta.

In [ ]:
import os
LOCAL, BACKUP = 'data/cuentos_colab', f'{DRIVE}/datos_cuentos_colab.tar'
if os.path.exists(f'{LOCAL}/processed/meta.json'):
    print('Los datos ya están listos en esta sesión.')
elif os.path.exists(BACKUP):
    print('Recuperando los datos desde Drive...')
    !tar -xf {BACKUP}
else:
    !mini-llm-data --config configs/colab.yaml
    !mini-llm-prepare --config configs/colab.yaml
    !tar -cf {BACKUP} {LOCAL}/processed {LOCAL}/raw/manifest.json
    print('Copia de los datos guardada en Drive.')
!du -sh {LOCAL}/processed

## 5. Entrenar (el paso largo)

Mira el **ETA** en las primeras líneas. Si es más de lo que puedes esperar, detén la celda (botón ■) y
ejecuta la celda opcional de abajo con menos iteraciones. Si Colab se desconecta, ejecuta otra vez las
celdas 1 a 4 y esta: continuará sola desde el último `last.pt`.

In [ ]:
import os
RUN = 'artifacts/checkpoints/mini-llm-colab'
resume = '--resume' if os.path.exists(f'{RUN}/last.pt') else ''
print('Continuando desde last.pt' if resume else 'Entrenamiento nuevo')
!mini-llm-train --config configs/colab.yaml {resume}

**Opcional:** empezar de cero con menos iteraciones (por ejemplo 8000) si el ETA era demasiado largo.

In [ ]:
# !mini-llm-train --config configs/colab.yaml --overwrite --max-iters 8000

## 6. Evaluar (perplejidad, líneas base, precisión)

In [ ]:
!mini-llm-eval --config configs/colab.yaml

## 7. Ajuste fino para el chat (SFT)

In [ ]:
!mini-llm-sft --config configs/chat_colab.yaml

## 8. Paquete para tu PC

Crea `mini-llm-colab/para_mi_pc.zip` en tu Drive con los modelos, los tokenizadores y las métricas.
Descárgalo desde drive.google.com y descomprímelo sobre la carpeta del proyecto en tu PC.

In [ ]:
!rm -f {DRIVE}/para_mi_pc.zip
!cd /content/mini-llm && zip -r -q {DRIVE}/para_mi_pc.zip \
    artifacts/checkpoints/mini-llm-colab/best.pt artifacts/checkpoints/mini-llm-colab/metrics.csv \
    artifacts/checkpoints/mini-llm-colab/loss.png artifacts/checkpoints/mini-llm-colab/config.yaml \
    artifacts/checkpoints/mini-llm-chat-colab/best.pt artifacts/checkpoints/mini-llm-chat-colab/metrics.csv \
    artifacts/checkpoints/mini-llm-chat-colab/loss.png artifacts/checkpoints/mini-llm-chat-colab/config.yaml \
    artifacts/checkpoints/mini-llm-chat-colab/sft_report.json \
    artifacts/tokenizer/cuentos_colab artifacts/tokenizer/chat_cuentos_colab \
    artifacts/eval/mini-llm-colab data/cuentos_colab/processed/meta.json
!ls -lh {DRIVE}/para_mi_pc.zip